# Track 1, Option A — Notebook 01: Supervised Fine-Tuning (5 LoRA trials)

SFT of `Qwen/Qwen3-0.6B-Base` on `HuggingFaceH4/ultrachat_200k` (technically
preprocessed subset). Runs **5 trials** varying LoRA rank/target-modules and training
args (LR, batch, epochs), evaluates each on the 15-prompt test set with a comprehensive
metric suite (**BLEU, chrF, ROUGE-L, BERTScore P/R/F1**, response length), and
**auto-selects the best** trial (val loss as tiebreaker).

**Kaggle setup:** GPU (T4 x2 or P100), Internet **ON**.

**Outputs (for the report):**
- `sft_results.csv` — per-trial summary (all metrics, per-difficulty breakdown, trainable params, GPU memory, training time)
- `sft_per_prompt.csv` — per-prompt scores for every trial
- `sft_training_log.csv` — loss curves; `sft_by_difficulty.csv`, `sft_by_type.csv` — breakdowns
- `sft_preprocessing_stats.json` — dataset filtering + token-length stats
- `best_sft_config.json`, `sft_sample_outputs.md`, and the best adapter at `/kaggle/working/best_sft_adapter`

In [1]:
%%capture
!pip install -q -U "transformers>=4.51.0" "trl>=0.15.0" "peft>=0.14.0" \
    "datasets>=3.2.0" "accelerate>=1.3.0" "evaluate>=0.4.3" \
    "sacrebleu>=2.4.3" "bert-score>=0.3.13" "sentencepiece>=0.2.0" \
    "rouge-score>=0.1.2"
# Kaggle ships an old torchao (0.10) that PEFT's LoRA dispatcher rejects, even
# though we never use it. Remove it to avoid the ImportError in get_peft_model().
!pip uninstall -y torchao

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"             # train on ONE GPU (T4 x2 'auto' sharding stalls)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce GPU fragmentation
import json, gc, random, time, numpy as np, pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import sacrebleu
from bert_score import score as _bertscore
from tqdm.auto import tqdm

# ---------------- Config ----------------
BASE_MODEL     = "Qwen/Qwen3-0.6B-Base"
SYSTEM_PROMPT  = "You are a helpful assistant. Answer concisely and directly."
MAX_NEW_TOKENS = 256
MAX_SEQ_LEN    = 512    # capped for T4 memory (151k-vocab logits are the bottleneck)
N_TRAIN        = 2000
N_VAL          = 300
SEED           = 42
OUT_DIR        = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
# T4 (Turing) / P100 (Pascal) have fp16 tensor cores but NO native bf16 -> bf16 is
# emulated and ~2x slower. Force fp16 so matmuls hit the tensor cores.
DTYPE  = torch.float16
USE_BF16 = False
print("dtype:", DTYPE)

CHATML_TEMPLATE = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

dtype: torch.float16


In [3]:
# ---------------- Test set + shared eval utilities ----------------
INLINE_TEST_SET = {"prompts": [
    {"id": 1, "type": "factual_short", "difficulty": "easy", "prompt": "Which planet is known as the Red Planet? Answer in one short sentence.", "gold": "Mars is known as the Red Planet."},
    {"id": 2, "type": "sentiment_classification", "difficulty": "easy", "prompt": "Classify the sentiment of this sentence as positive or negative: 'I absolutely love this movie.'", "gold": "The sentiment is positive."},
    {"id": 3, "type": "math_reasoning", "difficulty": "easy", "prompt": "A train travels 60 km in 1.5 hours. What is its average speed in km/h?", "gold": "The average speed is 40 km/h."},
    {"id": 4, "type": "list_constrained", "difficulty": "medium", "prompt": "List exactly three primary colors, separated by commas.", "gold": "Red, yellow, blue."},
    {"id": 5, "type": "rewriting_tone", "difficulty": "medium", "prompt": "Rewrite this sentence to be more formal: 'Hey, can you send me that file ASAP?'", "gold": "Could you please send me that file at your earliest convenience?"},
    {"id": 6, "type": "summarization_grounded", "difficulty": "medium", "prompt": "Summarize the following text in one sentence: 'The library will be closed on Monday for a public holiday. It will reopen on Tuesday at 9 a.m. with normal hours.'", "gold": "The library is closed Monday for a public holiday and reopens Tuesday at 9 a.m."},
    {"id": 7, "type": "extraction", "difficulty": "medium", "prompt": "Extract when the library reopens from this text: 'The library will be closed on Monday for a public holiday. It will reopen on Tuesday at 9 a.m. with normal hours.'", "gold": "Tuesday at 9 a.m."},
    {"id": 8, "type": "sentiment_3class", "difficulty": "medium", "prompt": "Classify the following review as positive, negative, or neutral: 'The food was okay, nothing special.'", "gold": "The sentiment is neutral."},
    {"id": 9, "type": "unit_conversion", "difficulty": "medium", "prompt": "Convert 2 kilometers to meters.", "gold": "2 kilometers is 2000 meters."},
    {"id": 10, "type": "coding_oneliner", "difficulty": "medium", "prompt": "Write a single line of Python that computes the sum of a list named nums.", "gold": "total = sum(nums)"},
    {"id": 11, "type": "grammar", "difficulty": "medium", "prompt": "Give the past tense of the verb 'run' in a single word.", "gold": "Ran."},
    {"id": 12, "type": "translation", "difficulty": "medium", "prompt": "Translate 'Good morning' into French.", "gold": "Bonjour."},
    {"id": 13, "type": "date_reasoning", "difficulty": "harder", "prompt": "If today is Monday, what day will it be in 3 days?", "gold": "It will be Thursday."},
    {"id": 14, "type": "multi_step_math", "difficulty": "harder", "prompt": "A pizza is cut into 8 slices. If 3 people each eat 2 slices, how many slices remain?", "gold": "Two slices remain."},
    {"id": 15, "type": "explanation_audience", "difficulty": "harder", "prompt": "Explain what a CPU does in exactly two sentences.", "gold": "A CPU carries out the instructions of a program. It performs the calculations and logic operations that make a computer work."},
]}

def load_test_set():
    for p in ["test_set.json", "/kaggle/input/test-set/test_set.json", "/kaggle/working/test_set.json"]:
        if os.path.exists(p):
            with open(p) as f: return json.load(f)
    return INLINE_TEST_SET

PROMPTS = load_test_set()["prompts"]

def load_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.chat_template = CHATML_TEMPLATE
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    return tok

def build_prompt(tokenizer, user_msg):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def _eos_ids(tok):
    ids = []
    im_end = tok.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end >= 0: ids.append(im_end)
    if tok.eos_token_id is not None: ids.append(tok.eos_token_id)
    return ids or None

@torch.no_grad()
def generate_response(model, tokenizer, user_msg, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(tokenizer, user_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=_eos_ids(tokenizer))
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

from rouge_score import rouge_scorer
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def _bertscore_prf(preds, refs):
    P, R, F1 = _bertscore(preds, refs, lang="en", rescale_with_baseline=True)
    return ([float(x) * 100 for x in P], [float(x) * 100 for x in R], [float(x) * 100 for x in F1])

def evaluate_model(model, tokenizer, prompts, tag="model"):
    """Generate on every prompt and score with a comprehensive metric suite.
    Returns a per-prompt DataFrame (one row per prompt, all metrics)."""
    preds, refs, rows = [], [], []
    for p in tqdm(prompts, desc=f'generate [{tag}]'):
        pred = generate_response(model, tokenizer, p["prompt"])
        preds.append(pred); refs.append(p["gold"])
        rows.append({"trial": tag, "id": p["id"], "type": p["type"],
                     "difficulty": p.get("difficulty", "NA"),
                     "prompt": p["prompt"], "gold": p["gold"], "prediction": pred,
                     "pred_words": len(pred.split()), "pred_chars": len(pred),
                     "gold_words": len(p["gold"].split())})
    bleus  = [sacrebleu.sentence_bleu(pr, [rf]).score for pr, rf in zip(preds, refs)]
    chrfs  = [sacrebleu.sentence_chrf(pr, [rf]).score for pr, rf in zip(preds, refs)]
    rouges = [_rouge.score(rf, pr)["rougeL"].fmeasure * 100 for pr, rf in zip(preds, refs)]
    bp, br, bf = _bertscore_prf(preds, refs)
    for i, r in enumerate(rows):
        r.update(bleu=round(bleus[i], 3), chrf=round(chrfs[i], 3), rougeL=round(rouges[i], 3),
                 bertscore_p=round(bp[i], 3), bertscore_r=round(br[i], 3), bertscore_f1=round(bf[i], 3))
    df = pd.DataFrame(rows)
    print(f"[{tag}] BLEU={df.bleu.mean():.2f} | chrF={df.chrf.mean():.2f} | "
          f"ROUGE-L={df.rougeL.mean():.2f} | BERT-F1={df.bertscore_f1.mean():.2f} | "
          f"avg_len={df.pred_words.mean():.0f}w")
    return df



print("done")

done


In [4]:
# ---------------- Load + technically preprocess the SFT dataset (ultrachat_200k) -------
# Preprocessing pipeline (each step logged for the report):
#   1. random subsample (fixed seed)         5. deduplicate by prompt
#   2. keep first user->assistant turn        6. token-length windowing [MIN_TOK, MAX_TOK]
#   3. whitespace normalization               7. render to ChatML `text`
#   4. drop empty / too-short responses        8. record token-length distribution
import re
from datasets import Dataset

tokenizer = load_tokenizer(BASE_MODEL)

MIN_RESP_CHARS = 20            # drop near-empty assistant turns
MIN_TOK, MAX_TOK = 16, MAX_SEQ_LEN

raw_train = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").shuffle(seed=SEED).select(range(min(N_TRAIN * 3, 60000)))
raw_val   = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft").shuffle(seed=SEED).select(range(min(N_VAL * 3, 6000)))

def _norm(s):
    return re.sub(r"\s+", " ", s or "").strip()

def build_examples(ds, n_target):
    texts, ntoks, seen = [], [], set()
    c = dict(raw=len(ds), bad_format=0, too_short=0, duplicates=0, length_filtered=0)
    for ex in ds:
        msgs = ex.get("messages") or []
        if len(msgs) < 2 or msgs[0]["role"] != "user" or msgs[1]["role"] != "assistant":
            c["bad_format"] += 1; continue
        user, resp = _norm(msgs[0]["content"]), _norm(msgs[1]["content"])
        if not user or len(resp) < MIN_RESP_CHARS:
            c["too_short"] += 1; continue
        key = user[:200].lower()
        if key in seen:
            c["duplicates"] += 1; continue
        turn = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user}, {"role": "assistant", "content": resp}]
        text = tokenizer.apply_chat_template(turn, tokenize=False, add_generation_prompt=False)
        nt = len(tokenizer(text, add_special_tokens=False)["input_ids"])
        if nt < MIN_TOK or nt > MAX_TOK:
            c["length_filtered"] += 1; continue
        seen.add(key); texts.append(text); ntoks.append(nt)
        if len(texts) >= n_target:
            break
    c["kept"] = len(texts)
    return texts, ntoks, c

tr_texts, tr_tok, tr_stats = build_examples(raw_train, N_TRAIN)
va_texts, va_tok, va_stats = build_examples(raw_val, N_VAL)
train_ds = Dataset.from_dict({"text": tr_texts})
val_ds   = Dataset.from_dict({"text": va_texts})

def _pct(a):
    a = np.array(a)
    return {"min": int(a.min()), "p25": int(np.percentile(a, 25)), "median": int(np.median(a)),
            "p75": int(np.percentile(a, 75)), "max": int(a.max()), "mean": round(float(a.mean()), 1)}

prep_stats = {"dataset": "HuggingFaceH4/ultrachat_200k",
              "train_filtering": tr_stats, "val_filtering": va_stats,
              "train_token_len": _pct(tr_tok), "val_token_len": _pct(va_tok),
              "filters": {"min_resp_chars": MIN_RESP_CHARS, "min_tokens": MIN_TOK,
                          "max_tokens": MAX_TOK, "dedup": "by lowercased prompt prefix",
                          "normalization": "collapse whitespace"}}
with open(os.path.join(OUT_DIR, "sft_preprocessing_stats.json"), "w") as f:
    json.dump(prep_stats, f, indent=2)
print(json.dumps(prep_stats, indent=2))
print("\ntrain:", len(train_ds), "| val:", len(val_ds))
print("---- example ----\n", train_ds[0]["text"][:600])

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

{
  "dataset": "HuggingFaceH4/ultrachat_200k",
  "train_filtering": {
    "raw": 6000,
    "bad_format": 0,
    "too_short": 2,
    "duplicates": 0,
    "length_filtered": 1532,
    "kept": 2000
  },
  "val_filtering": {
    "raw": 900,
    "bad_format": 0,
    "too_short": 1,
    "duplicates": 0,
    "length_filtered": 250,
    "kept": 300
  },
  "train_token_len": {
    "min": 47,
    "p25": 252,
    "median": 328,
    "p75": 411,
    "max": 512,
    "mean": 324.8
  },
  "val_token_len": {
    "min": 84,
    "p25": 233,
    "median": 323,
    "p75": 393,
    "max": 511,
    "mean": 314.2
  },
  "filters": {
    "min_resp_chars": 20,
    "min_tokens": 16,
    "max_tokens": 512,
    "dedup": "by lowercased prompt prefix",
    "normalization": "collapse whitespace"
  }
}

train: 2000 | val: 300
---- example ----
 <|im_start|>system
You are a helpful assistant. Answer concisely and directly.<|im_end|>
<|im_start|>user
How does the location of the Sydney Conservatorium of Music impact the

In [5]:
# ---------------- 5 SFT trials: LoRA + training-arg grid ----------------
ALL_LINEAR = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# bs = per-device batch (raised now that checkpointing is OFF + fp16); ga = grad-accum
# so the EFFECTIVE batch (bs*ga) is unchanged across trials: 4, 8, 8, 16, 16.
SFT_TRIALS = [
    {"name": "sft1", "r": 8,  "alpha": 16,  "targets": ["q_proj", "v_proj"],                     "lr": 2e-4, "bs": 4, "ga": 1, "epochs": 1},
    {"name": "sft2", "r": 16, "alpha": 32,  "targets": ["q_proj", "k_proj", "v_proj", "o_proj"], "lr": 2e-4, "bs": 4, "ga": 2, "epochs": 2},
    {"name": "sft3", "r": 32, "alpha": 64,  "targets": ALL_LINEAR,                                "lr": 1e-4, "bs": 2, "ga": 4, "epochs": 2},
    {"name": "sft4", "r": 16, "alpha": 32,  "targets": ALL_LINEAR,                                "lr": 3e-4, "bs": 4, "ga": 4, "epochs": 1},
    {"name": "sft5", "r": 64, "alpha": 128, "targets": ALL_LINEAR,                                "lr": 5e-5, "bs": 2, "ga": 8, "epochs": 3},
]

def make_sft_config(cfg, out):
    kw = dict(output_dir=out, per_device_train_batch_size=cfg["bs"],
              per_device_eval_batch_size=8, gradient_accumulation_steps=cfg["ga"],
              learning_rate=cfg["lr"], num_train_epochs=cfg["epochs"], logging_steps=25,
              eval_strategy="epoch", save_strategy="no", warmup_ratio=0.03,
              lr_scheduler_type="cosine", max_grad_norm=1.0,
              gradient_checkpointing=False,  # off: single GPU + 0.6B fits easily, ~30% faster
              bf16=USE_BF16, fp16=not USE_BF16, report_to="none", seed=SEED,
              dataset_text_field="text", packing=False)
    # `max_seq_length` was renamed to `max_length` in some TRL releases — handle both.
    try:
        return SFTConfig(max_seq_length=MAX_SEQ_LEN, **kw)
    except TypeError:
        return SFTConfig(max_length=MAX_SEQ_LEN, **kw)

def make_lora(cfg):
    return LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=0.05,
                      bias="none", task_type="CAUSAL_LM", target_modules=cfg["targets"])

def _tier(pp, difficulty, metric):
    sub = pp[pp["difficulty"] == difficulty]
    return round(float(sub[metric].mean()), 3) if len(sub) else float("nan")

def run_sft_trial(cfg):
    set_seed(SEED)
    gc.collect(); torch.cuda.empty_cache()  # free any leftover memory before loading
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    out = os.path.join(OUT_DIR, cfg["name"])
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE).to("cuda")
    model.config.use_cache = False
    model.enable_input_require_grads()  # required for gradient checkpointing + PEFT
    t0 = time.time()
    trainer = SFTTrainer(model=model, args=make_sft_config(cfg, out),
                         train_dataset=train_ds, eval_dataset=val_ds,
                         processing_class=tokenizer, peft_config=make_lora(cfg))
    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trainer.model.parameters())
    trainer.train()
    val_loss = float(trainer.evaluate().get("eval_loss", float("nan")))
    train_time = time.time() - t0
    peak_gb = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
    train_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    final_train_loss = float(train_losses[-1]) if train_losses else float("nan")
    log_df = pd.DataFrame(trainer.state.log_history); log_df.insert(0, "trial", cfg["name"])

    trainer.model.config.use_cache = True
    pp = evaluate_model(trainer.model, tokenizer, PROMPTS, tag=cfg["name"])
    trainer.save_model(out)

    res = {"name": cfg["name"], "r": cfg["r"], "alpha": cfg["alpha"],
           "targets": cfg["targets"] if isinstance(cfg["targets"], str) else ",".join(cfg["targets"]),
           "lr": cfg["lr"], "per_device_bs": cfg["bs"], "grad_accum": cfg["ga"],
           "eff_batch": cfg["bs"] * cfg["ga"], "epochs": cfg["epochs"],
           "trainable_params": trainable, "trainable_pct": round(100 * trainable / total, 4),
           "final_train_loss": round(final_train_loss, 4), "eval_loss": round(val_loss, 4),
           "bleu": round(float(pp.bleu.mean()), 3), "chrf": round(float(pp.chrf.mean()), 3),
           "rougeL": round(float(pp.rougeL.mean()), 3),
           "bertscore_p": round(float(pp.bertscore_p.mean()), 3),
           "bertscore_r": round(float(pp.bertscore_r.mean()), 3),
           "bertscore_f1": round(float(pp.bertscore_f1.mean()), 3),
           "combined": round(float(pp.bleu.mean() + pp.bertscore_f1.mean()), 3),
           "bleu_easy": _tier(pp, "easy", "bleu"), "bleu_medium": _tier(pp, "medium", "bleu"),
           "bleu_harder": _tier(pp, "harder", "bleu"),
           "bertf1_easy": _tier(pp, "easy", "bertscore_f1"), "bertf1_medium": _tier(pp, "medium", "bertscore_f1"),
           "bertf1_harder": _tier(pp, "harder", "bertscore_f1"),
           "avg_pred_words": round(float(pp.pred_words.mean()), 1),
           "peak_gpu_gb": round(peak_gb, 2), "train_time_s": round(train_time, 1),
           "adapter_dir": out}
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return res, pp, log_df



print("done")

done


In [6]:
# ---------------- Run all 5 SFT trials ----------------
results, per_prompt_all, logs_all = [], [], []
for i, cfg in enumerate(SFT_TRIALS, 1):
    print("\n" + "=" * 60 + f"\nSFT TRIAL {i}/{len(SFT_TRIALS)}: {cfg['name']}\n" + "=" * 60)
    res, pp, log_df = run_sft_trial(cfg)
    results.append(res); per_prompt_all.append(pp); logs_all.append(log_df)

sft_df = pd.DataFrame(results)
sft_df.to_csv(os.path.join(OUT_DIR, "sft_results.csv"), index=False)

# Per-prompt scores for every trial (enables per-difficulty / per-type analysis).
per_prompt_df = pd.concat(per_prompt_all, ignore_index=True)
per_prompt_df.to_csv(os.path.join(OUT_DIR, "sft_per_prompt.csv"), index=False)

# Training/eval loss curves for every trial.
pd.concat(logs_all, ignore_index=True).to_csv(os.path.join(OUT_DIR, "sft_training_log.csv"), index=False)
print("\nSaved: sft_results.csv, sft_per_prompt.csv, sft_training_log.csv")
sft_df


SFT TRIAL 1/5: sft1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

...

generate [sft5]:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[sft5] BLEU=14.25 | chrF=36.93 | ROUGE-L=35.11 | BERT-F1=18.12 | avg_len=60w

Saved: sft_results.csv, sft_per_prompt.csv, sft_training_log.csv


,name,r,alpha,targets,lr,per_device_bs,grad_accum,eff_batch,epochs,trainable_params,...,bleu_easy,bleu_medium,bleu_harder,bertf1_easy,bertf1_medium,bertf1_harder,avg_pred_words,peak_gpu_gb,train_time_s,adapter_dir
0,sft1,8,16,"q_proj,v_proj",0.00020,4,1,4,1,1146880,...,7.151,13.966,1.759,16.768,17.715,9.835,118.3,9.81,396.4,/kaggle/working/sft1
1,sft2,16,32,"q_proj,k_proj,v_proj,o_proj",0.00020,4,2,8,2,4587520,...,15.655,13.052,1.445,-4.115,-26.484,9.326,92.4,10.41,825.1,/kaggle/working/sft2
2,sft3,32,64,"q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,...",0.00010,2,4,8,2,20185088,...,8.964,9.895,1.514,9.905,5.496,12.385,96.4,8.91,889.7,/kaggle/working/sft3
3,sft4,16,32,"q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,...",0.00030,4,4,16,1,10092544,...,16.509,17.958,1.606,31.313,17.077,9.808,80.2,11.39,483.1,/kaggle/working/sft4
4,sft5,64,128,"q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,...",0.00005,2,8,16,3,40370176,...,12.032,19.229,1.525,26.650,18.253,9.198,60.4,9.15,1322.0,/kaggle/working/sft5


In [7]:
# ---------------- Select best SFT trial ----------------
# Primary: highest combined (BLEU + BERTScore). Tiebreaker (within 1.0 pt): lower val loss.
import shutil
ranked = sft_df.sort_values("combined", ascending=False).reset_index(drop=True)
top = ranked.iloc[0]
close = ranked[ranked["combined"] >= top["combined"] - 1.0]
best = close.sort_values("eval_loss", ascending=True).iloc[0]
best_name = best["name"]
print("Ranking by combined score:\n",
      ranked[["name", "bleu", "chrf", "rougeL", "bertscore_f1", "combined", "eval_loss", "train_time_s", "peak_gpu_gb"]])
print(f"\nBEST SFT TRIAL: {best_name} "
      f"(BLEU {best['bleu']}, BERTScore {best['bertscore_f1']}, val loss {best['eval_loss']})")

best_cfg = next(c for c in SFT_TRIALS if c["name"] == best_name)
best_cfg_serializable = {**best_cfg, "targets": best_cfg["targets"]}
with open(os.path.join(OUT_DIR, "best_sft_config.json"), "w") as f:
    json.dump({"trial": best_name, "config": best_cfg_serializable,
               "bleu": float(best["bleu"]), "bertscore_f1": float(best["bertscore_f1"]),
               "eval_loss": float(best["eval_loss"])}, f, indent=2)

# Copy the winning adapter to a stable path for Notebook 02.
best_adapter_dst = os.path.join(OUT_DIR, "best_sft_adapter")
if os.path.exists(best_adapter_dst):
    shutil.rmtree(best_adapter_dst)
shutil.copytree(best["adapter_dir"], best_adapter_dst)
print("Saved best adapter ->", best_adapter_dst)

Ranking by combined score:
    name    bleu    chrf  rougeL  bertscore_f1  combined  eval_loss  \
0  sft4  14.398  35.779  33.564        18.470    32.868     1.6297   
1  sft5  14.249  36.935  35.113        18.121    32.370     1.6368   
2  sft1  10.161  27.489  22.498        15.950    26.111     1.6383   
3  sft3   8.032  26.010  26.192         7.755    15.788     1.6275   
4  sft2  11.251  18.841  24.349       -14.848    -3.597     1.6284   

   train_time_s  peak_gpu_gb  
0         483.1        11.39  
1        1322.0         9.15  
2         396.4         9.81  
3         889.7         8.91  
4         825.1        10.41  

BEST SFT TRIAL: sft4 (BLEU 14.398, BERTScore 18.47, val loss 1.6297)
Saved best adapter -> /kaggle/working/best_sft_adapter


In [8]:
# ---------------- Save example outputs + aggregate breakdowns for the report ----------------
best_pp = per_prompt_df[per_prompt_df["trial"] == best_name].reset_index(drop=True)

with open(os.path.join(OUT_DIR, "sft_sample_outputs.md"), "w") as f:
    f.write(f"# SFT sample outputs (best trial: {best_name})\n\n")
    for _, r in best_pp.iterrows():
        f.write(f"**Prompt ({r['type']}, {r['difficulty']}):** {r['prompt']}\n\n")
        f.write(f"- Gold: {r['gold']}\n")
        f.write(f"- SFT (BLEU {r['bleu']}, ROUGE-L {r['rougeL']}, BERT-F1 {r['bertscore_f1']}): {r['prediction']}\n\n")

# Mean metrics per (trial, difficulty) — drives the by-difficulty figure/table.
by_diff = (per_prompt_df.groupby(["trial", "difficulty"])[["bleu", "chrf", "rougeL", "bertscore_f1"]]
           .mean().round(3).reset_index())
by_diff.to_csv(os.path.join(OUT_DIR, "sft_by_difficulty.csv"), index=False)

# Mean metrics per (trial, type) — finer-grained breakdown.
by_type = (per_prompt_df.groupby(["trial", "type"])[["bleu", "rougeL", "bertscore_f1"]]
           .mean().round(3).reset_index())
by_type.to_csv(os.path.join(OUT_DIR, "sft_by_type.csv"), index=False)

print("Done. Files saved to", OUT_DIR, ":")
for fn in ["sft_results.csv", "sft_per_prompt.csv", "sft_training_log.csv",
           "sft_by_difficulty.csv", "sft_by_type.csv", "sft_preprocessing_stats.json",
           "best_sft_config.json", "sft_sample_outputs.md", "best_sft_adapter/"]:
    print("  -", fn)

Done. Files saved to /kaggle/working :
  - sft_results.csv
  - sft_per_prompt.csv
  - sft_training_log.csv
  - sft_by_difficulty.csv
  - sft_by_type.csv
  - sft_preprocessing_stats.json
  - best_sft_config.json
  - sft_sample_outputs.md
  - best_sft_adapter/
